## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: TRANSFORMACIÓN (CAPA SILVER)
**OBJETIVO:** Limpiar, cruzar tablas (Tickets + Agentes) y aplicar reglas de negocio para enriquecer los datos.

In [0]:
from pyspark.sql.functions import col, current_timestamp, when
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
    
# Limpiamos widgets previos para evitar conflictos
dbutils.widgets.removeAll()

In [0]:
# 1. PARÁMETROS DINÁMICOS (WIDGETS)

dbutils.widgets.text("catalogo", "catalogo_pqrs")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

print("✅ Parámetros cargados.")

In [0]:
# 2. LECTURA DE TABLAS DESDE BRONZE

df_tickets = spark.table(f"{catalogo}.{esquema_source}.tickets_raw")
df_agentes = spark.table(f"{catalogo}.{esquema_source}.agentes_soporte_raw")

In [0]:
display(df_tickets)

In [0]:
display(df_agentes)

In [0]:
# 3. CREACIÓN DE FUNCIÓN PERSONALIZADA (UDF)
# Regla de negocio: Categorizar la criticidad basada en palabras clave del cuerpo del mensaje

def evaluar_criticidad(mensaje):
    # Si el mensaje viene vacío o nulo, le asignamos Baja por defecto
    if not mensaje:
        return "Baja"
        
    mensaje_lower = mensaje.lower()
    
    # Palabras clave para criticidad Alta
    palabras_alta = ["urgente", "emergencia", "grave", "caído", "inmediato", "inmediata", "roto"]
    if any(palabra in mensaje_lower for palabra in palabras_alta):
        return "Alta"
        
    # Palabras clave para criticidad Media
    palabras_media = ["ayuda", "duda", "problema", "revisar", "falla", "error"]
    if any(palabra in mensaje_lower for palabra in palabras_media):
        return "Media"
        
    # Si no tiene palabras clave críticas, es Baja
    return "Baja"

criticidad_udf = F.udf(evaluar_criticidad, StringType())

In [0]:
# 4. LIMPIEZA Y TRANSFORMACIÓN
# Eliminamos nulos SOLO en ticket_id (permitimos agentes nulos si el ticket es nuevo y no ha sido asignado)
# Hacemos cast de agente_id a int para asegurar el cruce más adelante

df_tickets_clean = df_tickets.dropna(how="any", subset=["ticket_id"]) \
    .withColumn("agente_id", col("agente_id").cast("int")) \
    .withColumn("criticidad", criticidad_udf(col("cuerpo_mensaje")))

In [0]:
display(df_tickets_clean)

In [0]:
# 5. CRUCE DE TABLAS (JOIN)
# EL TRUCO INFALIBLE: Estandarizar la tabla agentes en memoria antes de cruzar
# Si en Producción la columna aún se llama 'id_agente', la renombramos "al vuelo"
if "id_agente" in df_agentes.columns:
    df_agentes = df_agentes.withColumnRenamed("id_agente", "agente_id")

# Aseguramos que el agente_id en la tabla agentes también sea INT
df_agentes = df_agentes.withColumn("agente_id", col("agente_id").cast("int"))

# Cruzamos los tickets con la información del agente asignado
df_joined = df_tickets_clean.alias("t").join(
    df_agentes.alias("a"),
    col("t.agente_id") == col("a.agente_id"),
    "left"
)

In [0]:
display(df_joined)

In [0]:
# 6. SELECCIÓN FINAL Y GUARDADO EN SILVER

tabla_destino = f"{catalogo}.{esquema_sink}.tickets_enriquecidos"

df_silver = df_joined.select(
    col("t.ticket_id"),
    col("t.cliente_id"),
    col("t.cuerpo_mensaje").alias("descripcion"),
    col("criticidad"),
    col("a.nombre_completo").alias("agente_asignado"),
    col("a.region").alias("region_agente"),
    col("a.nivel_soporte"),
    current_timestamp().alias("fecha_transformacion")
)

df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_destino)

print(f"✅ Transformación completada. Tabla guardada en {tabla_destino}")

In [0]:
display(tabla_destino)

In [0]:
display(df_silver)